# 面试题：不用 Trainer，怎样实现一个可解释的文档分类器？

## 可以直接复述的回答

文档分类要先明确标签边界、训练与测试切分，再把文本转换为固定维度特征，最后用可度量的损失学习决策边界。关键词规则适合做可解释基线，但对同义表达和多词组合很脆弱。一个线性 softmax 分类器已经足以展示完整训练机制：词袋向量乘权重得到 logits，log-sum-exp 得到交叉熵，再通过 backward 计算真实梯度并手动更新参数。评估不能只看总准确率，还应逐样本查看概率、预测和错误原因。分类器遇到全 OOV 文本时仍会被 bias 强迫给出某类，因此需要覆盖率门禁或拒识类别。本题用退款、发票、账号三类客服工单，从零训练 PyTorch 参数并打印 loss、梯度和每类高权重词。

## 真实案例

数据模拟工单标题，训练集 15 条、测试集 6 条，每条都有人工意图标签。文本已人工空格分词以隔离模型机制；这是小型教学实验，不能外推真实客服准确率。

In [1]:
from pprint import pprint  # 导入结构化打印函数以展示工单和结果
import torch  # 导入 PyTorch 以执行真实张量前向与反向传播
torch.manual_seed(7)  # 固定随机种子保证教学输出可复现
torch.set_num_threads(1)  # 限制线程数以减少小实验运行抖动
labels = ["退款问题", "发票问题", "账号问题"]  # 定义三个互斥业务标签
train_records = [{"text": "退款 到账 太慢", "label": "退款问题"}, {"text": "商品 破损 申请 退款", "label": "退款问题"}, {"text": "售后 退货 钱款 进度", "label": "退款问题"}, {"text": "订单 退钱 进度 查询", "label": "退款问题"}, {"text": "退款 凭证 上传 照片", "label": "退款问题"}, {"text": "电子 发票 下载", "label": "发票问题"}, {"text": "公司 抬头 修改", "label": "发票问题"}, {"text": "开票 税号 错误", "label": "发票问题"}, {"text": "发票 邮箱 未收到", "label": "发票问题"}, {"text": "订单 开具 票据", "label": "发票问题"}, {"text": "密码 忘记 无法 登录", "label": "账号问题"}, {"text": "账号 被锁 解冻", "label": "账号问题"}, {"text": "手机 换绑 失败", "label": "账号问题"}, {"text": "登录 验证码 收不到", "label": "账号问题"}, {"text": "账户 安全 验证", "label": "账号问题"}]  # 构造十五条分布均衡的训练工单
test_records = [{"text": "退款 进度 查询", "label": "退款问题"}, {"text": "破损 商品 退钱", "label": "退款问题"}, {"text": "税号 抬头 修改", "label": "发票问题"}, {"text": "票据 邮箱 下载", "label": "发票问题"}, {"text": "账号 登录 被锁", "label": "账号问题"}, {"text": "验证码 手机 换绑", "label": "账号问题"}]  # 构造六条包含同义组合的测试工单
print("训练工单预览：")  # 输出真实案例标题
pprint(train_records)  # 展示训练文本和人工标签
print("测试工单预览：")  # 输出测试集标题
pprint(test_records)  # 展示六条同数据评估样本

训练工单预览：
[{'label': '退款问题', 'text': '退款 到账 太慢'},
 {'label': '退款问题', 'text': '商品 破损 申请 退款'},
 {'label': '退款问题', 'text': '售后 退货 钱款 进度'},
 {'label': '退款问题', 'text': '订单 退钱 进度 查询'},
 {'label': '退款问题', 'text': '退款 凭证 上传 照片'},
 {'label': '发票问题', 'text': '电子 发票 下载'},
 {'label': '发票问题', 'text': '公司 抬头 修改'},
 {'label': '发票问题', 'text': '开票 税号 错误'},
 {'label': '发票问题', 'text': '发票 邮箱 未收到'},
 {'label': '发票问题', 'text': '订单 开具 票据'},
 {'label': '账号问题', 'text': '密码 忘记 无法 登录'},
 {'label': '账号问题', 'text': '账号 被锁 解冻'},
 {'label': '账号问题', 'text': '手机 换绑 失败'},
 {'label': '账号问题', 'text': '登录 验证码 收不到'},
 {'label': '账号问题', 'text': '账户 安全 验证'}]
测试工单预览：
[{'label': '退款问题', 'text': '退款 进度 查询'},
 {'label': '退款问题', 'text': '破损 商品 退钱'},
 {'label': '发票问题', 'text': '税号 抬头 修改'},
 {'label': '发票问题', 'text': '票据 邮箱 下载'},
 {'label': '账号问题', 'text': '账号 登录 被锁'},
 {'label': '账号问题', 'text': '验证码 手机 换绑'}]


## Baseline / 基线：少量关键词规则

规则只识别“退款、发票、密码”三个显式词，其他表达默认进入账号类。它容易解释，却会漏掉“退钱、票据、抬头”等同义表达。

In [2]:
def keyword_baseline(text):  # 定义只有三个关键词的规则基线
    if "退款" in text:  # 检查最显式的退款关键词
        return "退款问题"  # 命中后返回退款类别
    if "发票" in text:  # 检查最显式的发票关键词
        return "发票问题"  # 命中后返回发票类别
    if "密码" in text:  # 检查最显式的账号关键词
        return "账号问题"  # 命中后返回账号类别
    return "账号问题"  # 未命中时使用脆弱的默认类别
baseline_rows = [{"text": record["text"], "真实": record["label"], "预测": keyword_baseline(record["text"])} for record in test_records]  # 对六条测试工单执行规则分类
baseline_accuracy = sum(row["真实"] == row["预测"] for row in baseline_rows) / len(baseline_rows)  # 计算规则基线准确率
print("关键词 Baseline 的逐样本结果：")  # 输出基线结果标题
pprint(baseline_rows)  # 展示规则在同一测试集上的每条决策
print(f"Baseline accuracy={baseline_accuracy:.3f}")  # 输出基线聚合指标

关键词 Baseline 的逐样本结果：
[{'text': '退款 进度 查询', '真实': '退款问题', '预测': '退款问题'},
 {'text': '破损 商品 退钱', '真实': '退款问题', '预测': '账号问题'},
 {'text': '税号 抬头 修改', '真实': '发票问题', '预测': '账号问题'},
 {'text': '票据 邮箱 下载', '真实': '发票问题', '预测': '账号问题'},
 {'text': '账号 登录 被锁', '真实': '账号问题', '预测': '账号问题'},
 {'text': '验证码 手机 换绑', '真实': '账号问题', '预测': '账号问题'}]
Baseline accuracy=0.500


## 特征工程：手写词表与词袋矩阵

词表只从训练集构建，避免测试泄漏。每行词袋向量表示一条工单，每列表示一个训练词；当前实现保留词频，生产上可再比较二值特征、TF-IDF 或子词特征。

In [3]:
vocabulary = sorted({token for record in train_records for token in record["text"].split()})  # 只用训练集构造稳定词表
token_to_index = {token: index for index, token in enumerate(vocabulary)}  # 建立 token 到特征列的映射
label_to_index = {label: index for index, label in enumerate(labels)}  # 建立业务标签到类别编号的映射
def vectorize(records):  # 定义手写词袋向量化函数
    matrix = torch.zeros((len(records), len(vocabulary)), dtype=torch.float32)  # 创建样本数乘词表大小的零矩阵
    for row_index, record in enumerate(records):  # 遍历每条文本记录
        for token in record["text"].split():  # 遍历人工分词后的 token
            if token in token_to_index:  # 只编码训练词表内的已知 token
                matrix[row_index, token_to_index[token]] += 1.0  # 在对应词袋位置累加词频
    return matrix  # 返回可送入线性模型的特征矩阵
train_x = vectorize(train_records)  # 向量化十五条训练工单
train_y = torch.tensor([label_to_index[record["label"]] for record in train_records], dtype=torch.long)  # 把训练标签转换为整数张量
test_x = vectorize(test_records)  # 用相同词表向量化六条测试工单
test_y = torch.tensor([label_to_index[record["label"]] for record in test_records], dtype=torch.long)  # 把测试标签转换为整数张量
print(f"词表大小={len(vocabulary)}, train_x shape={tuple(train_x.shape)}, test_x shape={tuple(test_x.shape)}")  # 输出关键张量形状
print("第一条工单的非零词袋特征：", {vocabulary[index]: float(value) for index, value in enumerate(train_x[0]) if value > 0})  # 展示一条可读特征向量

词表大小=44, train_x shape=(15, 44), test_x shape=(6, 44)
第一条工单的非零词袋特征： {'到账': 1.0, '太慢': 1.0, '退款': 1.0}


## 手写模型与真实 forward/backward

下面不用 nn.Linear 和优化器封装：权重、bias、softmax 交叉熵和 SGD 更新都显式写出。第一步梯度范数证明 loss 确实沿计算图反传到参数，而不是只做一次推理。

In [4]:
class TinyTextClassifier(torch.nn.Module):  # 定义最小可训练线性文本分类器
    def __init__(self, input_size, class_count):  # 初始化权重矩阵和偏置向量
        super().__init__()  # 初始化 PyTorch 模块基类
        self.weight = torch.nn.Parameter(torch.randn(input_size, class_count) * 0.02)  # 创建词特征到类别的可训练权重
        self.bias = torch.nn.Parameter(torch.zeros(class_count))  # 创建每个类别的可训练偏置
    def forward(self, features):  # 定义从词袋到 logits 的前向传播
        return features @ self.weight + self.bias  # 执行显式矩阵乘法和广播加法
def manual_cross_entropy(logits, targets):  # 定义数值稳定的多分类交叉熵
    log_normalizer = torch.logsumexp(logits, dim=1, keepdim=True)  # 用 log-sum-exp 计算归一化项
    log_probabilities = logits - log_normalizer  # 得到每个类别的对数概率
    selected = log_probabilities[torch.arange(targets.shape[0]), targets]  # 取出每条样本真实类别的对数概率
    return -selected.mean()  # 返回批次平均负对数似然
model = TinyTextClassifier(len(vocabulary), len(labels))  # 实例化手写线性分类器
learning_rate = 0.2  # 设置小数据全批量梯度下降步长
training_ledger = []  # 保存损失曲线与第一步梯度中间量
for epoch in range(201):  # 对完整训练集执行二百零一次参数更新
    logits = model(train_x)  # 运行真实 forward 得到训练 logits
    loss = manual_cross_entropy(logits, train_y)  # 用手写交叉熵计算标量损失
    loss.backward()  # 运行真实 backward 计算参数梯度
    gradient_norm = float(model.weight.grad.norm())  # 读取权重梯度范数作为训练证据
    if epoch % 40 == 0:  # 每四十轮记录一次而不输出冗长日志
        training_ledger.append({"epoch": epoch, "loss": round(float(loss), 5), "weight_grad_norm": round(gradient_norm, 5)})  # 保存损失与梯度曲线点
    with torch.no_grad():  # 关闭梯度记录以手动执行 SGD 更新
        model.weight -= learning_rate * model.weight.grad  # 沿负梯度方向更新权重
        model.bias -= learning_rate * model.bias.grad  # 沿负梯度方向更新偏置
    model.weight.grad.zero_()  # 清零权重梯度避免跨轮累加
    model.bias.grad.zero_()  # 清零偏置梯度避免跨轮累加
print("训练 loss 与真实梯度范数：")  # 输出训练过程标题
pprint(training_ledger)  # 展示 loss 下降及梯度变化

训练 loss 与真实梯度范数：
[{'epoch': 0, 'loss': 1.10962, 'weight_grad_norm': 0.42754},
 {'epoch': 40, 'loss': 0.3796, 'weight_grad_norm': 0.19449},
 {'epoch': 80, 'loss': 0.20078, 'weight_grad_norm': 0.11167},
 {'epoch': 120, 'loss': 0.13193, 'weight_grad_norm': 0.0759},
 {'epoch': 160, 'loss': 0.09705, 'weight_grad_norm': 0.05682},
 {'epoch': 200, 'loss': 0.07634, 'weight_grad_norm': 0.04517}]


## 结果表与结果解读

线性模型会把“退钱、破损、税号、票据、被锁、换绑”等训练中出现的词分别推向对应类别。逐样本概率比单个准确率更有信息：如果最高概率与次高概率接近，生产系统应降低自动处理权限。

In [5]:
def probabilities_from_logits(logits):  # 定义不调用封装 softmax 的概率转换
    shifted = logits - logits.max(dim=1, keepdim=True).values  # 减去行最大值避免指数溢出
    exponentials = shifted.exp()  # 对稳定后的 logits 逐元素取指数
    return exponentials / exponentials.sum(dim=1, keepdim=True)  # 按行归一化得到类别概率
with torch.no_grad():  # 关闭评估阶段梯度记录
    test_logits = model(test_x)  # 对六条测试工单运行前向传播
    test_probabilities = probabilities_from_logits(test_logits)  # 把 logits 转换为可解释概率
    predicted_indices = test_probabilities.argmax(dim=1)  # 选择概率最高的类别编号
model_rows = []  # 创建逐样本模型结果表
for row_index, record in enumerate(test_records):  # 遍历每条测试工单
    probability_map = {labels[class_index]: round(float(test_probabilities[row_index, class_index]), 3) for class_index in range(len(labels))}  # 构造三类概率映射
    model_rows.append({"text": record["text"], "真实": record["label"], "预测": labels[int(predicted_indices[row_index])], "概率": probability_map})  # 保存文本、标签、预测和概率
model_accuracy = sum(row["真实"] == row["预测"] for row in model_rows) / len(model_rows)  # 计算模型测试准确率
top_tokens = {}  # 创建每个类别的高权重词解释
for class_index, label in enumerate(labels):  # 遍历三个输出类别
    ranked_indices = torch.argsort(model.weight[:, class_index], descending=True)[:5]  # 找出最支持当前类别的五个词
    top_tokens[label] = [(vocabulary[int(index)], round(float(model.weight[int(index), class_index]), 3)) for index in ranked_indices]  # 保存词和对应线性权重
print("逐样本模型结果：")  # 输出结果表标题
pprint(model_rows)  # 展示每条工单的三类概率和预测
print(f"模型 accuracy={model_accuracy:.3f}，Baseline={baseline_accuracy:.3f}")  # 输出同一测试集上的指标对照
print("每类最高权重词：")  # 输出模型机制解释标题
pprint(top_tokens)  # 展示模型从训练数据学到的类别词证据

逐样本模型结果：
[{'text': '退款 进度 查询',
  '概率': {'发票问题': 0.014, '账号问题': 0.017, '退款问题': 0.97},
  '真实': '退款问题',
  '预测': '退款问题'},
 {'text': '破损 商品 退钱',
  '概率': {'发票问题': 0.124, '账号问题': 0.13, '退款问题': 0.746},
  '真实': '退款问题',
  '预测': '退款问题'},
 {'text': '税号 抬头 修改',
  '概率': {'发票问题': 0.911, '账号问题': 0.051, '退款问题': 0.038},
  '真实': '发票问题',
  '预测': '发票问题'},
 {'text': '票据 邮箱 下载',
  '概率': {'发票问题': 0.893, '账号问题': 0.066, '退款问题': 0.041},
  '真实': '发票问题',
  '预测': '发票问题'},
 {'text': '账号 登录 被锁',
  '概率': {'发票问题': 0.032, '账号问题': 0.945, '退款问题': 0.023},
  '真实': '账号问题',
  '预测': '账号问题'},
 {'text': '验证码 手机 换绑',
  '概率': {'发票问题': 0.056, '账号问题': 0.902, '退款问题': 0.042},
  '真实': '账号问题',
  '预测': '账号问题'}]
模型 accuracy=1.000，Baseline=0.500
每类最高权重词：
{'发票问题': [('发票', 1.051),
          ('票据', 0.685),
          ('开具', 0.676),
          ('错误', 0.632),
          ('修改', 0.622)],
 '账号问题': [('登录', 0.965),
          ('安全', 0.671),
          ('解冻', 0.663),
          ('换绑', 0.656),
          ('账号', 0.634)],
 '退款问题': [('退款', 1.42),
          ('进度

## 失败案例：全 OOV 文本仍被强制分类

“身份认证 人脸失败”中的词训练时从未出现，词袋因此全为零；模型只能依赖 bias 给出一个看似合法的类别，这不是有证据的判断。修正是计算已知词覆盖率，低于阈值就拒识或转人工；生产上还可使用字符/子词特征并收集新类数据。

In [6]:
oov_record = {"text": "身份认证 人脸失败", "label": "未知新意图"}  # 构造训练词表完全未覆盖的新业务工单
oov_x = vectorize([oov_record])  # 用旧词表向量化失败样本
with torch.no_grad():  # 关闭失败演示中的梯度记录
    oov_probabilities = probabilities_from_logits(model(oov_x))[0]  # 获取模型被迫给出的三类概率
forced_label = labels[int(oov_probabilities.argmax())]  # 读取无文本证据时的强制类别
raw_tokens = oov_record["text"].split()  # 拆分原始输入以计算词表覆盖率
known_ratio = sum(token in token_to_index for token in raw_tokens) / len(raw_tokens)  # 计算已知 token 占比
fixed_decision = forced_label if known_ratio >= 0.5 else "转人工：词表覆盖不足"  # 用覆盖率门禁阻止无证据自动分类
print("失败案例：全零词袋仍输出类别", {"特征和": float(oov_x.sum()), "强制类别": forced_label, "概率": [round(float(value), 3) for value in oov_probabilities]})  # 展示错误行为及其根因
print("修正后的决策：", {"known_ratio": known_ratio, "decision": fixed_decision})  # 展示覆盖率门禁的处理结果

失败案例：全零词袋仍输出类别 {'特征和': 0.0, '强制类别': '发票问题', '概率': [0.242, 0.386, 0.372]}
修正后的决策： {'known_ratio': 0.0, 'decision': '转人工：词表覆盖不足'}


## 生产差距

真实工单通常是多标签、长尾且随业务漂移，需要更可靠的分词或子词编码、时间切分、类别不平衡处理和概率校准。上线还要定义拒识阈值、人工回流、敏感文本脱敏、模型/词表版本、分群混淆矩阵与漂移告警；本例的全批量 CPU 训练只用于解释 forward/backward。

In [7]:
assert train_x.shape == (15, len(vocabulary))  # 验证十五条训练工单被映射到统一词表空间
assert training_ledger[-1]["loss"] < training_ledger[0]["loss"]  # 验证真实反向传播使训练损失下降
assert model_accuracy > baseline_accuracy  # 验证学习模型优于同测试集关键词基线
assert model_accuracy == 1.0  # 验证六条教学测试样本均正确分类
assert float(oov_x.sum()) == 0.0  # 验证失败样本确实完全超出训练词表
assert fixed_decision == "转人工：词表覆盖不足"  # 验证覆盖率门禁拒绝无证据预测
print("最小回归测试通过：前向、反向、逐样本分类与 OOV 拒识均满足预期")  # 输出集中断言的验收结论

最小回归测试通过：前向、反向、逐样本分类与 OOV 拒识均满足预期
